In [25]:
import pandas as pd
import re
from pathlib import Path
from urllib.parse import urlparse

In [26]:
import pandas as pd
import re
from pathlib import Path

main_file = Path(r"D:\Proyek FEB\Publikasi internasional.xlsx")
sinta_file = Path(r"D:\Proyek FEB\lengkapin data\sinta_q_citation_results.xlsx")

df = pd.read_excel(main_file)
sinta = pd.read_excel(sinta_file)

COL_TITLE = "Title"
COL_Q = "SCOPUS Q"
COL_CITATION = "SCOPUS SITASI"

In [27]:
def is_empty_value(value):
    if pd.isna(value):
        return True
    
    text = str(value).strip()
    
    if text == "":
        return True
    
    if text.lower() in [
        "nan", "none", "null", "-", "?", "??", "???",
        "n/a", "na", "tidak ada", "not found"
    ]:
        return True
    
    return False


def normalize_title(value):
    if pd.isna(value):
        return ""
    
    text = str(value).lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", "", text)
    return text.strip()

In [28]:
# hanya pakai hasil SINTA yang ditemukan
sinta_found = sinta[sinta["status"] == "found"].copy()

df_updated = df.copy()

df_updated["title_key"] = df_updated[COL_TITLE].apply(normalize_title)
sinta_found["title_key"] = sinta_found["input_title"].apply(normalize_title)

result_map = (
    sinta_found
    .drop_duplicates(subset=["title_key"])
    .set_index("title_key")
    .to_dict(orient="index")
)

In [29]:
for idx, row in df_updated.iterrows():
    key = row["title_key"]
    
    if key not in result_map:
        continue
    
    result = result_map[key]
    
    # Isi SCOPUS Q hanya jika kosong / ?
    if is_empty_value(row.get(COL_Q)):
        q_value = result.get("scopus_q")
        
        if not is_empty_value(q_value):
            df_updated.at[idx, COL_Q] = q_value
    
    # Isi SCOPUS SITASI hanya jika kosong / ?
    if is_empty_value(row.get(COL_CITATION)):
        citation_value = result.get("scopus_citation")
        
        if not is_empty_value(citation_value):
            df_updated.at[idx, COL_CITATION] = int(citation_value)

In [30]:
df_updated = df_updated.drop(columns=["title_key"])

In [31]:
before_missing = df[[COL_Q, COL_CITATION]].applymap(is_empty_value).sum()
after_missing = df_updated[[COL_Q, COL_CITATION]].applymap(is_empty_value).sum()

comparison = pd.DataFrame({
    "before": before_missing,
    "after": after_missing,
    "filled": before_missing - after_missing
})

comparison

C:\Users\user\AppData\Local\Temp\ipykernel_2276\2247834279.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  before_missing = df[[COL_Q, COL_CITATION]].applymap(is_empty_value).sum()
C:\Users\user\AppData\Local\Temp\ipykernel_2276\2247834279.py:2: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  after_missing = df_updated[[COL_Q, COL_CITATION]].applymap(is_empty_value).sum()


,before,after,filled
SCOPUS Q,2153,1524,629
SCOPUS SITASI,2153,1524,629


In [32]:
df_updated.to_excel("Publikasi_internasional_updated_q_citation.xlsx", index=False)